# Lab 65: The Framing Effect

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-65.ipynb)

**What you will do:** run your own gain-versus-loss framing survey, compare it with the book's related medical-framing figures, pool the class's data, plot the prospect-theory value function, change its two parameters, test whether a small language model reverses its choice when the frame flips, and use the bet from Lab 66 to estimate how loss-averse you are.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 65 you gave half a group of people the **Asian disease problem**'s gain-framed version
("Programme A saves 200 people for certain; Programme B has a 1/3 chance of saving all 600 and a
2/3 chance of saving no one") and the other half the loss-framed version ("Programme C results
in 400 deaths for certain; Programme D has a 1/3 chance no one dies and a 2/3 chance all 600
die"). Programmes A and C are the same outcome; so are B and D. Most people choose the safe
option (A) under the gain frame but the risky option (D) under the loss frame — identical facts,
reversed preferences — because the frame sets the brain's reference point, and the prospect-theory
value function is concave for gains but convex for losses (Tversky & Kahneman, 1981). Loss
aversion — losses generally weighing more heavily than equivalent gains — is a related but
separate feature of the value function, explored in Lab 66.

## Record your results

Enter how many people in each frame chose the safe programme versus the risky programme.

In [ ]:
results = pd.DataFrame({
    "frame": ["gain (A vs B)", "loss (C vs D)"],
    "chose_safe": [7, 3],     # example data: replace with your own counts
    "chose_risky": [3, 7],    # example data
})
results["total"] = results.chose_safe + results.chose_risky
results["pct_safe"] = results.chose_safe / results.total
display(results)
results.plot.bar(x="frame", y="pct_safe", legend=False, color=["steelblue", "firebrick"], figsize=(4.5, 3))
plt.ylabel("proportion choosing the safe programme"); plt.ylim(0, 1); plt.xticks(rotation=0); plt.show()

## Compare

The lab does not give a percentage for this exact disease-programme scenario, only that "most
people" choose the safe option in the gain frame and the risky option in the loss frame. The
lab does give a percentage for a related medical framing (McNeil, Pauker, Sox, & Tversky,
1982): choosing surgery fell from 82% under a survival frame to 56% under a mortality frame.
That is a different scenario (surgery vs radiotherapy, not lives saved vs lost), so use it only
as context for how large this kind of reversal can be, not as a target your own numbers should
match.

In [ ]:
gain_pct = results.set_index("frame").loc["gain (A vs B)", "pct_safe"]
loss_pct = results.set_index("frame").loc["loss (C vs D)", "pct_safe"]
print(f"Your gain-frame safe-choice rate: {gain_pct:.0%}   (book: 'most people' choose safe)")
print(f"Your loss-frame safe-choice rate: {loss_pct:.0%}   (book: 'most people' choose risky, i.e. low safe-rate)")
print(f"For context, McNeil et al. (1982) found a related medical framing shift 82% -> 56%.")
if gain_pct > loss_pct:
    print("Your data shows the predicted reversal: more people chose the safe option under the")
    print("gain frame than under the loss frame.")
else:
    print("Your data did not show the predicted reversal. With a small group this can happen by")
    print("chance -- the effect is well replicated on average, not guaranteed for any one group.")

## Simulation

This lab showed that "90% survival" feels different from "10% mortality" even though the information is identical. This simulation plots the prospect theory value function (Kahneman & Tversky, 1979), using the diminishing-sensitivity and loss-aversion parameters later fitted from choice data (Tversky & Kahneman, 1992). The function explains why the two framings feel different: losses loom larger than gains, and the curve is concave for gains but convex for losses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-100, 100, 1000)

# Prospect theory value function (parameters from Tversky & Kahneman, 1992)
alpha = 0.88   # diminishing sensitivity
lamb = 2.25    # loss aversion coefficient


def value(x, alpha=0.88, lamb=2.25):
    magnitude = np.abs(x) ** alpha
    return np.where(x >= 0, magnitude, -lamb * magnitude)


v = value(x)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, v, "steelblue", linewidth=2.5)
ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="grey", linewidth=0.5)
ax.fill_between(x[x >= 0], 0, v[x >= 0], alpha=0.1, color="green")
ax.fill_between(x[x < 0], 0, v[x < 0], alpha=0.1, color="red")

# Annotate key features
ax.annotate(
    "Gains: concave\n(diminishing joy)",
    xy=(60, value(np.array([60]))[0]),
    fontsize=9, ha="center", color="green",
)
ax.annotate(
    "Losses: convex and steeper\n(amplified pain)",
    xy=(-60, value(np.array([-60]))[0]),
    fontsize=9, ha="center", color="red",
)

# Show asymmetry: value(50) vs value(-50)
gain50 = value(np.array([50]))[0]
loss50 = value(np.array([-50]))[0]
ax.plot([50, 50], [0, gain50], "g--", alpha=0.5)
ax.plot([-50, -50], [0, loss50], "r--", alpha=0.5)
ax.text(55, gain50 / 2, f"+50 → {gain50:.1f}", fontsize=8, color="green")
ax.text(-95, loss50 / 2, f"-50 → {loss50:.1f}", fontsize=8, color="red")

ax.set_xlabel("Objective outcome (gain or loss)")
ax.set_ylabel("Subjective value")
ax.set_title("Prospect Theory: Why Losses Hurt More Than Gains Feel Good")
plt.tight_layout()
plt.show()

ratio = abs(loss50) / gain50
print(f"Gaining 50 feels like: +{gain50:.1f}")
print(f"Losing 50 feels like:  {loss50:.1f}")
print(f"Loss/gain ratio: {ratio:.1f}x")
print(f"-> losses loom {ratio:.1f} times larger")

**What to notice:** The curve is steeper on the left (losses) than on the right (gains). With these parameters, losing 50 units produces a little over twice as much subjective pain as gaining 50 units produces pleasure. This asymmetry is why framing the same outcome as a loss or a gain changes your decision—even when the objective information is identical. Treat the exact 2:1 figure as one influential estimate rather than a universal constant: later work has found the ratio varies a good deal with context, and some analyses find much weaker asymmetry than this classic curve suggests [→ Lab 66].

## Change the parameters

Move the sliders. **α** (alpha) controls diminishing sensitivity: below 1, each extra pound
matters a little less. **λ** (lambda) controls loss aversion: how much more a loss weighs than
an equal gain. The text below the plot evaluates a 50–50 bet: win *G* or lose *L*.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def explore(alpha=0.88, lamb=2.25, win=12, lose=10):
    xs = np.linspace(-100, 100, 1001)
    vs = value(xs, alpha, lamb)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(xs, vs, color="steelblue", linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.axvline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("Objective outcome (£)")
    ax.set_ylabel("Subjective value")
    ax.set_ylim(-160, 70)
    ax.set_title(f"alpha = {alpha:.2f}, lambda = {lamb:.2f}")
    plt.show()
    v_bet = 0.5 * value(np.array([win]), alpha, lamb)[0] + 0.5 * value(np.array([-lose]), alpha, lamb)[0]
    print(f"Bet: 50% win £{win}, 50% lose £{lose}.  Expected money: £{0.5 * win - 0.5 * lose:+.2f}")
    print(f"Prospect-theory value of the bet: {v_bet:+.2f} ->", "ACCEPT" if v_bet > 0 else "DECLINE")

interact(explore,
         alpha=FloatSlider(value=0.88, min=0.3, max=1.0, step=0.02),
         lamb=FloatSlider(value=2.25, min=1.0, max=3.5, step=0.05),
         win=IntSlider(value=12, min=5, max=40),
         lose=IntSlider(value=10, min=5, max=40));

## The AI side

The book notes that some language models "still show framing-consistent preferences" even
though they can explain that the two versions are logically identical. Below, a small open
model (flan-t5-small) is given both frames of the disease problem, and its choices are then
checked against what the prospect-theory `value()` function from the simulation above would
itself recommend, using the same alpha and lambda parameters.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-small")
gain_frame = (
    "A disease is expected to kill 600 people. Programme A will save 200 people for certain. "
    "Programme B has a 1/3 probability of saving all 600 people and a 2/3 probability of "
    "saving no one. Which programme do you choose? Answer with just the letter, A or B.")
loss_frame = (
    "A disease is expected to kill 600 people. Programme C will result in 400 deaths for "
    "certain. Programme D has a 1/3 probability that no one dies and a 2/3 probability that "
    "all 600 die. Which programme do you choose? Answer with just the letter, C or D.")
model_gain_choice = qa(gain_frame, max_new_tokens=5)[0]["generated_text"].strip()
model_loss_choice = qa(loss_frame, max_new_tokens=5)[0]["generated_text"].strip()
print(f"Model's choice, gain frame (A = safe, B = gamble): {model_gain_choice}")
print(f"Model's choice, loss frame (C = safe, D = gamble): {model_loss_choice}")

# What would the value() function above itself recommend, using the same 600-life scenario,
# scaled onto the same -100..+100 range the simulation plots?
scale = 600 / 100
safe_gain = value(np.array([200 / scale]))[0]
risky_gain = (1 / 3) * value(np.array([600 / scale]))[0] + (2 / 3) * value(np.array([0.0]))[0]
safe_loss = value(np.array([-400 / scale]))[0]
risky_loss = (1 / 3) * value(np.array([0.0]))[0] + (2 / 3) * value(np.array([-600 / scale]))[0]
print(f"\nProspect-theory value, gain frame: safe = {safe_gain:.1f}, gamble = {risky_gain:.1f} "
      f"-> theory recommends {'safe (A)' if safe_gain > risky_gain else 'gamble (B)'}")
print(f"Prospect-theory value, loss frame: safe = {safe_loss:.1f}, gamble = {risky_loss:.1f} "
      f"-> theory recommends {'safe (C)' if safe_loss > risky_loss else 'gamble (D)'}")

With the standard alpha = 0.88, lambda = 2.25 parameters, the value() function itself
reproduces the classic reversal -- it favours the safe option in the gain frame and the
gamble in the loss frame, purely from the shape of the curve you explored above. Whether
flan-t5-small's own two answers match this pattern is a separate question: a small model can
answer confidently without weighing the options the way prospect theory describes, so treat
its choices as a rough, sometimes inconsistent illustration, not proof that it "feels" the
framing effect the way you do.

## Pool the class data

Pool the class's safe-vs-risky choice counts by frame (anonymous IDs, not names).

In [ ]:
import io
csv_text = """id,frame,chose_safe,chose_risky
P01,gain,6,2
P01,loss,2,6
P02,gain,7,3
P02,loss,3,7
P03,gain,5,3
P03,loss,4,6
"""  # example data: replace with your class CSV, e.g. pd.read_csv("framing_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
summary = df.groupby("frame")[["chose_safe", "chose_risky"]].sum()
summary["pct_safe"] = summary.chose_safe / (summary.chose_safe + summary.chose_risky)
display(summary)
boot = {f: [df.loc[df.frame == f].sample(frac=1, replace=True).chose_safe.sum()
            / (df.loc[df.frame == f].sample(frac=1, replace=True).chose_safe.sum()
               + df.loc[df.frame == f].sample(frac=1, replace=True).chose_risky.sum())
            for _ in range(3000)] for f in ["gain", "loss"]}
gap = np.array(boot["gain"]) - np.array(boot["loss"])
lo, hi = np.percentile(gap, [2.5, 97.5])
print(f"Class pooled gap (gain safe-rate minus loss safe-rate): {gap.mean():.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%})")
print("For context, McNeil et al. (1982) found an analogous, differently framed medical shift")
print("from 82% to 56% -- a 26-point reversal.")

## Questions to think about

1. Set λ to 1.0. Is the £12/£10 bet now accepted? What does λ = 1 mean psychologically?
2. Keep λ = 2.25 and raise the win until the bet is accepted. How large must it be? Compare this with the "about £20" threshold described in Lab 66.
3. Set α to 1.0 and then to 0.5. What changes about how people treat large versus small amounts?
4. Lab 66 reports that the "2:1" figure is debated (Gal & Rucker, 2018). Which slider would you move to represent a person who is barely loss-averse?

## Challenge

**Estimate your own loss aversion.** In Lab 66 you found the smallest win *G* that would make you
accept a 50–50 bet against a £10 loss. At that point the gamble feels neutral, so
0.5·G^α = 0.5·λ·10^α, which gives λ = (G / 10)^α. Enter your own *G* below (and, if you like,
collect values from classmates) to see where you fall compared with the textbook value of 2.25.

In [ ]:
my_G = 20          # the smallest win (in £) you would accept against a £10 loss
alpha = 0.88
my_lambda = (my_G / 10) ** alpha
print(f"Your estimated loss-aversion coefficient: lambda = {my_lambda:.2f}")

# Optional: pool answers from a class (edit this list) and compare
class_G = [15, 18, 20, 20, 22, 25, 30, 12, 20, 28]
lambdas = [(g / 10) ** alpha for g in class_G]
plt.figure(figsize=(6, 3))
plt.hist(lambdas, bins=8, color="steelblue", edgecolor="white")
plt.axvline(2.25, color="red", linestyle="--", label="Tversky & Kahneman (1992): 2.25")
plt.axvline(my_lambda, color="black", label="you")
plt.xlabel("lambda"); plt.ylabel("people"); plt.legend(); plt.show()
print("The class_G list above is example data; replace it with real answers.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-65.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")